<a href="https://colab.research.google.com/github/dcruzcavalieri/agentes-2026-2-modelo/blob/main/enc02_primeira_chamada_daniel.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Encontro 2 — Sua primeira chamada, e o ambiente que vamos usar o semestre todo

Tópicos Especiais em IA — Agentes Inteligentes · IFES Serra · 2026/2

---

## O que você entrega ao final desta aula

1. Este notebook rodando de ponta a ponta, **salvo no repositório da sua equipe**
2. O consumo de tokens da sua primeira chamada, anotado
3. A mesma pergunta respondida por **dois modelos diferentes**, sem alterar o código

## Antes de começar

No painel esquerdo, ícone de chave (**Secrets**), confirme que existem e estão liberados para este notebook:

- `GROQ_API_KEY` — provedor principal da disciplina
- `GEMINI_API_KEY` — usaremos só no Encontro 9, mas já deixamos pronto

Se faltar alguma, avise agora.

## Duas trilhas

Cada laboratório tem **caminho mínimo** e **caminho estendido**. O mínimo é obrigatório e tem o código quase pronto — você preenche o que está marcado com `# SEU CÓDIGO`. O estendido é para quem terminar antes; não vale nota, vale aprendizado.

## Parte 0 — Célula de preparo

**Toda aula começa por aqui.** A máquina virtual do Colab é apagada entre sessões: o que você instalou ontem não está mais aqui. Por isso todo notebook precisa se reconstruir sozinho.

Hoje instalamos **uma biblioteca só**: o cliente da OpenAI. E vamos usá-lo para conversar com um provedor que **não** é a OpenAI — o que é justamente o ponto da Parte 3.

A versão está limitada com `<3`, e não é preciosismo: bibliotecas desta área quebram a interface entre versões maiores. No Encontro 10, quando entrar um *framework* de agentes, você vai ver a versão fixada com `==` exatamente por isso.

In [1]:
%pip install -q "openai>=1.99.0,<3"

import importlib.metadata as md
print("openai", md.version("openai"))

openai 2.45.0


## Parte 1 — A chave nunca entra no código

Chave de API é como senha. Se ela for para o repositório, qualquer pessoa com acesso pode gastar a sua cota — e apagar o arquivo depois não resolve, porque o histórico guarda tudo.

No Colab, a chave vive nos **Secrets**: pertence à sua conta, você libera por notebook, e ela **não** vai junto quando você compartilha o notebook.

A função abaixo também funciona fora do Colab, lendo variável de ambiente. É um exemplo pequeno de código que roda em dois contextos — ideia que volta várias vezes no semestre.

In [2]:
import os

def obter_chave(nome: str) -> str:
    """Le um segredo dos Secrets do Colab; fora do Colab, da variavel de ambiente."""
    try:
        from google.colab import userdata
        return userdata.get(nome)
    except ImportError:
        valor = os.getenv(nome)
        if not valor:
            raise RuntimeError(f"Defina {nome} nos Secrets do Colab ou no ambiente.")
        return valor

GROQ_KEY = obter_chave("GROQ_API_KEY")

# Confirma que a chave chegou, sem imprimi-la. Nunca imprima uma chave inteira.
print("chave carregada, termina em:", GROQ_KEY[-4:])

chave carregada, termina em: 1uUo


## Parte 2 — Python que você vai usar o semestre todo

Quatro coisas, e não mais que isso. Se você já programa em Python, pule para a Parte 3.

| Construção | Onde aparece nesta disciplina |
|---|---|
| **função com anotação de tipo** | toda ferramenta do agente é uma função anotada |
| ***docstring*** | é o texto que o modelo lê para decidir se usa a ferramenta |
| **dicionário** | mensagens, parâmetros e respostas de API são dicionários |
| **JSON** | o formato em que tudo isso viaja pela rede |

A *docstring* merece atenção: no Encontro 5 você vai descobrir que ela **não é comentário** — é a descrição que o modelo lê para escolher a ferramenta. Uma *docstring* ruim é uma ferramenta que o agente nunca chama.

In [3]:
import json

def temperatura_reator(reator: str) -> str:
    """Le a temperatura atual de um reator da planta.

    Args:
        reator: identificador do reator, por exemplo R-101
    """
    leituras = {"R-101": 87.4, "R-102": 91.2, "R-103": 78.9}
    if reator not in leituras:
        return f"reator {reator} desconhecido"
    return f"{reator}: {leituras[reator]} graus Celsius"

print(temperatura_reator("R-101"))
print(temperatura_reator("R-999"))

# Um dicionario, e o mesmo dicionario em JSON — o formato em que ele viaja pela rede.
mensagem = {"role": "user", "content": "Qual a temperatura do reator R-101?"}
print(mensagem["content"])
print(json.dumps(mensagem, ensure_ascii=False))

R-101: 87.4 graus Celsius
reator R-999 desconhecido
Qual a temperatura do reator R-101?
{"role": "user", "content": "Qual a temperatura do reator R-101?"}


## Parte 3 — Sua primeira chamada

Repare no que **não** está no código abaixo: o nome do provedor. Só uma URL, uma chave e um nome de modelo — todos em variáveis. A biblioteca se chama `openai`, e estamos falando com o Groq.

Isso se chama **desacoplamento de fornecedor**, e não é firula. Ao preparar esta disciplina, em 28/07/2026, o provedor que seria o principal foi reprovado num teste de uso de ferramentas. A troca custou duas linhas — e é por isso que a abstração existe.

In [4]:
from openai import OpenAI

# --- as tres linhas que definem o provedor ---
LLM_BASE_URL = "https://api.groq.com/openai/v1"
LLM_MODEL = "llama-3.1-8b-instant"
LLM_API_KEY = GROQ_KEY
# ---------------------------------------------

cliente = OpenAI(base_url=LLM_BASE_URL, api_key=LLM_API_KEY)

resposta = cliente.chat.completions.create(
    model=LLM_MODEL,
    messages=[
        {"role": "system", "content": "Você responde a engenheiros. Seja preciso e breve."},
        {"role": "user", "content": "Em uma frase: o que diferencia um agente de um chatbot?"},
    ],
)

print(resposta.choices[0].message.content)

Um agente, em teoria de sistemas e comunicação, é um termo amplo que se refere a um sistema ou programa capaz de interpretar e agir em resposta a um determinado estímulo, podendo interagir e aprender, enquanto um chatbot é uma variante específica de agente programado para realizar tarefas específicas, geralmente interagindo com humanos por meio de interfaces de mensagem, como chatbots de atendimento ao cliente ou assistentes virtuais.


### Leia o consumo — é a conta que você vai pagar

Todo retorno traz quantos tokens foram gastos. **Anote esse número.** No Encontro 14 você vai comparar com o consumo de um agente que raciocina em vários passos, e a diferença entre os dois é o argumento daquela aula.

In [6]:
u = resposta.usage
print(f"entrada : {u.prompt_tokens} tokens")
print(f"saida   : {u.completion_tokens} tokens")
print(f"total   : {u.total_tokens} tokens")

TOKENS_PERGUNTA_SIMPLES = u.total_tokens
print("\nGuarde este numero para comparar no Encontro 14.")

entrada : 66 tokens
saida   : 106 tokens
total   : 172 tokens

Guarde este numero para comparar no Encontro 14.


## Parte 4 — Caminho mínimo: faça você

**Este é o entregável do laboratório.** Escreva uma função que faça uma pergunta ao modelo e devolva a resposta e o total de tokens. Preencha onde está marcado.

Se travar, o código da Parte 3 tem tudo o que você precisa — é copiar e adaptar.

In [7]:
def perguntar(pergunta: str, instrucao: str = "Você responde a engenheiros. Seja breve."):
    """Envia uma pergunta ao modelo e devolve (texto_da_resposta, total_de_tokens)."""
    r = cliente.chat.completions.create(
        model=LLM_MODEL,
        messages=[
            {"role": "system", "content": instrucao},
            # SEU CÓDIGO: acrescente a mensagem do usuario, com a pergunta recebida
            {"role": "user", "content": pergunta},
        ],
    )
    texto = None    # SEU CÓDIGO: extraia o texto da resposta
    tokens = None   # SEU CÓDIGO: extraia o total de tokens

    texto = r.choices[0].message.content
    tokens = r.usage.total_tokens

    return texto, tokens


# Teste: as tres perguntas devem responder, e nenhuma deve imprimir None
for p in ("O que é PEAS?",
          "Cite um risco de usar agentes em malha de controle em tempo real.",
          "O modelo executa a ferramenta, ou apenas pede que ela seja executada?"):
    texto, tokens = perguntar(p)
    print(f"[{tokens} tokens] {p}\n  -> {texto}\n")

[104 tokens] O que é PEAS?
  -> PEAS é um acrônimo que significa Proposta, Exigência, Ação e Solução, usado frequentemente em Engenharia de Produto para organizar ideias e soluções de problemas durante o design de produtos.

[181 tokens] Cite um risco de usar agentes em malha de controle em tempo real.
  -> Um risco de usar agentes em malha de controle em tempo real é a **incerteza na simulação do comportamento dos processos**. 

Agentes em malha de controle em tempo real são baseados em regras e procedimentos predeterminados que podem não capturar os comportamentos imprevisíveis e dinâmicos dos processos reais, levando a erros e instabilidades na simulação e controle. Isso pode afetar a precisão e a eficácia da malha de controle.

[336 tokens] O modelo executa a ferramenta, ou apenas pede que ela seja executada?
  -> Em um modelo de engenharia, existem diferentes abordagens para implementar a execução de uma ferramenta. Aqui estão algumas opções:

1. **Invocação direta**: O modelo inv

## Parte 5 — A temperatura, e por que nada disso é determinístico

O modelo sorteia o próximo token de uma distribuição de probabilidade. A `temperature` controla o quanto esse sorteio se espalha: perto de zero ele fica repetitivo e previsível; alto, criativo e instável.

Rode a célula e **compare as saídas**. Este é o problema central do Ciclo 3: como avaliar um sistema que não repete a si mesmo.

In [9]:
PERGUNTA = "Dê um nome curto para um agente que diagnostica falhas em uma planta industrial."

for temp in (0.0, 0.0, 1.2, 1.2):
    r = cliente.chat.completions.create(
        model=LLM_MODEL,
        messages=[{"role": "user", "content": PERGUNTA}],
        temperature=temp,
        max_tokens=30,
    )
    print(f"temperatura {temp}: {r.choices[0].message.content.strip()}\n")

print("\nAs duas primeiras linhas tendem a coincidir; as duas ultimas, nao.")
print("Guarde a pergunta: como se testa um programa que nao repete a si mesmo?")

temperatura 0.0: Um nome curto para um agente que diagnostica falhas em uma planta industrial pode ser:

- "FaultFinder" (encontrador

temperatura 0.0: Um nome curto para um agente que diagnostica falhas em uma planta industrial pode ser:

- "FaultFinder" (encontrador

temperatura 1.2: Você pode chamar esse agente de:

- FAID (Agente de Diagnóstico de Falhas)
- PID (Agente de

temperatura 1.2: Alguns nomes possíveis podem ser:

- Monitor
- Auditor
- Inspect
- SupervisAR
- Vigilante
-


As duas primeiras linhas tendem a coincidir; as duas ultimas, nao.
Guarde a pergunta: como se testa um programa que nao repete a si mesmo?


## Parte 6 — Trocar de modelo sem tocar no código

O exercício central da aula. Abaixo, duas configurações do mesmo provedor: um modelo pequeno e um grande. O código que faz a pergunta é **idêntico** nos dois casos.

Compare três coisas: a **qualidade** da resposta, o **consumo de tokens** e o **tempo**. Essa escolha de trade-off volta no Encontro 14.

In [10]:
import time

CONFIGURACOES = {
    "pequeno": dict(base_url="https://api.groq.com/openai/v1",
                    model="llama-3.1-8b-instant", key=GROQ_KEY),
    "grande":  dict(base_url="https://api.groq.com/openai/v1",
                    model="llama-3.3-70b-versatile", key=GROQ_KEY),
}

PERGUNTA = ("Um operador relata vibração anormal na bomba P-204. "
            "Liste no máximo três hipóteses de causa, em ordem de probabilidade.")

for nome, cfg in CONFIGURACOES.items():
    try:
        c = OpenAI(base_url=cfg["base_url"], api_key=cfg["key"])
        t0 = time.time()
        r = c.chat.completions.create(
            model=cfg["model"],
            messages=[{"role": "user", "content": PERGUNTA}],
            temperature=0.0,
        )
        dt = time.time() - t0
        print(f"=== {nome}: {cfg['model']} | {r.usage.total_tokens} tokens | {dt:.1f}s ===")
        print(r.choices[0].message.content.strip(), "\n")
    except Exception as e:
        print(f"=== {nome} FALHOU: {type(e).__name__}: {str(e)[:160]}\n")

print("O modelo maior costuma organizar melhor, e custa mais tokens e mais tempo.")

=== pequeno: llama-3.1-8b-instant | 363 tokens | 0.6s ===
Aqui estão três hipóteses de causa para a vibração anormal na bomba P-204, em ordem de probabilidade:

1. **Desalinhamento ou deslocamento do eixo da bomba**: Isso pode ocorrer devido a vibrações externas, como movimentos de equipamentos próximos ou mudanças no nível de fluido. É uma hipótese provável, pois é uma causa comum de vibrações anormais em bombas.

2. **Desgaste ou corrosão de componentes**: O desgaste ou corrosão de componentes, como a rota de entrada ou saída da bomba, pode causar vibrações anormais. Isso pode ocorrer devido à exposição a fluidos corrosivos ou ao desgaste normal com o tempo.

3. **Problemas de balanceamento ou equilíbrio da bomba**: Se a bomba não estiver balanceada corretamente, pode causar vibrações anormais. Isso pode ocorrer devido a problemas de fabricação ou ajustes incorretos durante a instalação.

É importante notar que essas hipóteses devem ser investigadas e verificadas por um técnico quali

## Parte 7 — Os erros que vão acontecer

Você está usando serviço gratuito e compartilhado. Falhar faz parte, e em geral não é culpa sua.

| Código | O que significa | O que fazer |
|---|---|---|
| `401` | chave inválida | gere outra e atualize o Secret |
| `429` | passou do limite por minuto | espere e repita |
| `503` | o modelo está saturado | troque de modelo ou espere |
| `400` | requisição malformada | leia a mensagem: em geral é parâmetro errado |

A célula abaixo provoca um `401` **de propósito**, para você reconhecer a mensagem quando ela aparecer de verdade. No Encontro 3 o seu código passa a tratar `429` e `503` sozinho, com repetição e espera crescente.

In [11]:
try:
    OpenAI(base_url=LLM_BASE_URL, api_key="chave-invalida-de-proposito").chat.completions.create(
        model=LLM_MODEL, messages=[{"role": "user", "content": "oi"}]
    )
except Exception as e:
    print("tipo:", type(e).__name__)
    print("mensagem:", str(e)[:220])
    print("\nEra o esperado. Reconhecer a mensagem economiza tempo depois.")

tipo: AuthenticationError
mensagem: Error code: 401 - {'error': {'message': 'Invalid API Key', 'type': 'invalid_request_error', 'code': 'invalid_api_key'}}

Era o esperado. Reconhecer a mensagem economiza tempo depois.


## Parte 8 — Caminho estendido (opcional)

Para quem terminou antes. Nenhum vale nota; todos voltam mais adiante.

1. **Meça o custo em moeda.** Busque o preço por milhão de tokens do modelo que você usou e calcule quanto custou a Parte 6. Depois estime o custo de rodar 200 vezes por dia, durante um mês.
2. **Force um `429`.** Faça vinte chamadas em sequência e veja o limite aparecer. Anote em qual chamada aconteceu.
3. **Compare instruções de sistema.** Rode a mesma pergunta com três instruções — uma vaga, uma específica, uma com exemplo — e observe o efeito. É o Encontro 4 antecipado.
4. **Reescreva `perguntar` com repetição.** Se der `429` ou `503`, espere 2 s, depois 4 s, depois 8 s. É o Encontro 3 antecipado.

In [ ]:
# Espaco livre para o caminho estendido.


## Parte 9 — Salvar no repositório da equipe

Não há comando a digitar. No menu do Colab:

**Arquivo → Salvar uma cópia no GitHub**

1. Autorize o Colab a acessar sua conta do GitHub — pedido uma única vez
2. Escolha o repositório da sua equipe
3. Caminho do arquivo: `notebooks/enc02_<seu-primeiro-nome>.ipynb`
4. Mensagem do *commit*: `encontro 2: primeira chamada e troca de modelo`
5. **Confirme no navegador** que o arquivo apareceu no repositório

**Um notebook por pessoa, com o seu nome no arquivo.** Assim ninguém sobrescreve o trabalho de ninguém, e o histórico mostra o que cada um fez — uma das evidências da nota de participação.

### Verificação final

Antes de sair da aula:

- [ ] o notebook rodou de ponta a ponta, sem erro, começando pela célula de preparo
- [ ] a Parte 4 está preenchida e as três perguntas responderam, sem `None`
- [ ] a Parte 6 mostrou os dois modelos respondendo, com tokens e tempo
- [ ] o consumo da Parte 3 está anotado
- [ ] o notebook está no repositório, com o seu nome no arquivo
- [ ] nenhuma chave aparece no código

**Teste que separa quem entendeu:** vá em *Ambiente de execução → Desconectar e excluir ambiente de execução* e rode tudo de novo. Se falhar, alguma célula depende de algo que não está no notebook — e é isso que significa reprodutibilidade.